# Análisis integrado de candidatos a círculos de hadas

**Alta Guajira, Colombia. Sentinel-2: compuesto 2024.**

Este cuaderno analiza las exportaciones de GEE_01, GEE_02 y GEE_03. Genera
tablas de contrastes, correlaciones de Spearman, cruces con la revisión visual,
figuras de morfometría y un mapa interactivo. Conserva los candidatos y las
categorías de prioridad originales.

## 1. Preparar el entorno

In [ ]:
#@title Comprobar bibliotecas
import importlib.util
import subprocess
import sys

requeridas = ['numpy', 'pandas', 'matplotlib', 'seaborn']
faltantes = [p for p in requeridas if importlib.util.find_spec(p) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *faltantes])
print('Bibliotecas disponibles. Ejecuta la celda del motor.')


In [ ]:
#@title Motor de análisis (ejecutar una vez; abrir código para consultar funciones)
"""Análisis descriptivo reproducible de candidatos a círculos de hadas.

Entradas: exportaciones GEE_01, GEE_02 y GEE_03. Se conservan las
clasificaciones originales. El módulo usa pandas, numpy, matplotlib y seaborn.
Las geometrías GeoJSON se cruzan por id_cand y se interpretan en WGS84.
"""
from __future__ import annotations

import argparse
import hashlib
import html
import importlib.metadata
import io
import json
import math
import platform
import re
import unicodedata
import zipfile
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

VERSION = "1.1.0"
INDICES = ["NDVI", "MSAVI", "GLI", "MSI", "BSI"]
INDICES_15 = ["NDVI", "EVI", "GNDVI", "CIG", "OSAVI", "ARVI", "GLI",
              "MSAVI", "SAVI", "MSI", "GVMI", "NDWI", "BSI", "NBR", "NDBI"]
PRIORIDADES = ["Muy alta", "Alta", "Media", "Baja", "Sin señal fuerte"]
CONFIANZAS = ["Alta", "Media", "Baja", "Sin información"]
COLOR_PRIORIDAD = {"Muy alta": "#b2182b", "Alta": "#ef8a62", "Media": "#d8a51b",
                   "Baja": "#4393c3", "Sin señal fuerte": "#969696"}
COLOR_CONFIANZA = {"Alta": "#1b7837", "Media": "#d8b365", "Baja": "#737373",
                  "Sin información": "#d9d9d9"}
COLOR_GRUPO = {"ANH": "#277b8e", "QGIS": "#c48136"}
ARCHIVOS_DEFAULT = {
    "candidatos": "candidatos.csv",
    "contrastes_5": "contrastes_5.csv",
    "contrastes_15": "contrastes_15.csv",
    "distribuciones_5": "distribuciones_5.csv",
    "circularidad_gee03": "circularidad_gee03.csv",
    "geometria": "candidatos.geojson",
    "area_estudio": "area_estudio.geojson",
    "poligonos_locales": "poligonos_locales.geojson",
    "anh_geometria": "anh.geojson",
}
CONTEXTO_DEFAULT = {
    "periodo_sentinel2": "2024", "anillo_entorno_m": 30,
    "escala_analisis_m": 20, "resolucion_ortoimagen_m": 0.5,
    "fecha_insumos_ortoimagen": "2023-01-19",
    "nota_periodo": "Periodo declarado según los scripts GEE; los CSV no incluyen fechas de escenas.",
    "superficies_sector_km2": {},
}
NOTAS_METODO = [
    "Los contrastes son media interior menos media del anillo exterior.",
    "La ANH es un conjunto de referencia exploratorio; no constituye una verdad de terreno de emisiones de H₂.",
    "La confianza visual y la prioridad espectral se conservan como clasificaciones diferentes.",
    "Se mantienen todos los candidatos. Los filtros del mapa modifican únicamente la visualización.",
    "Los cuartiles exportados de GEE se conservan para auditar sus reglas. Los cuantiles nuevos de Colab usan interpolación lineal.",
    "Se exportan desviaciones estándar poblacional (ddof=0) y muestral (ddof=1), identificadas por separado.",
    "Las correlaciones de Spearman son descriptivas y se calculan por grupo. No se interpretan como causalidad o validación de emisiones.",
    "Las geometrías circulares del GeoJSON representan áreas de muestreo, no contornos originales detectados.",
    "Los contrastes sin identificador se analizan por grupo; no se cruzan por orden de fila ni por radio.",
]


def token(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip()
    return " ".join(unicodedata.normalize("NFKD", text).encode("ascii", "ignore")
                    .decode().lower().split())


def confianza_normalizada(value: Any) -> str:
    k = token(value)
    mapping = {"alta": "Alta", "alto": "Alta", "media": "Media", "medio": "Media",
               "baja": "Baja", "bajo": "Baja", "": "Sin información",
               "sin informacion": "Sin información"}
    return mapping.get(k, str(value).strip())


def prioridad_normalizada(value: Any) -> str:
    mapping = {token(v): v for v in PRIORIDADES}
    if token(value) not in mapping:
        raise ValueError(f"Prioridad espectral no reconocida: {value!r}. Revisar la exportación GEE_02.")
    return mapping[token(value)]


@dataclass
class Auditoria:
    filas: list[dict[str, Any]] = field(default_factory=list)

    def agregar(self, control: str, estado: str, detalle: str, n: int = 0) -> None:
        self.filas.append({"control": control, "estado": estado,
                           "n_afectados": int(n), "detalle": detalle})

    def comprobar(self, control: str, cumple: bool, detalle: str, n: int = 0) -> None:
        self.agregar(control, "OK" if cumple else "ERROR", detalle, n)

    def tabla(self) -> pd.DataFrame:
        return pd.DataFrame(self.filas, columns=["control", "estado", "n_afectados", "detalle"])

    def detener_si_errores(self) -> None:
        errores = [v for v in self.filas if v["estado"] == "ERROR"]
        if errores:
            detalles = "\n".join(f"• {v['control']}: {v['detalle']}" for v in errores)
            raise ValueError("Los archivos requieren revisión antes de consolidar el análisis:\n" + detalles)


@dataclass
class Datos:
    candidatos: pd.DataFrame
    contrastes_5: pd.DataFrame | None
    contrastes_15: pd.DataFrame | None
    distribuciones_5: pd.DataFrame | None
    circularidad_gee03: pd.DataFrame | None
    geojson: dict | None
    capas_contexto: dict[str, dict]
    auditoria: Auditoria
    fuentes: list[dict]
    contexto: dict


def leer_csv(ruta: Path) -> pd.DataFrame:
    """Lee exportaciones GEE y CSV separados por punto y coma de Excel."""
    raw = ruta.read_bytes()
    try:
        contenido = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        contenido = raw.decode("latin-1")
    encabezado = next((v for v in contenido.splitlines() if v.strip()), "")
    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    tabla = pd.read_csv(io.StringIO(contenido), sep=sep, decimal="," if sep == ";" else ".")
    tabla.columns = [str(v).strip() for v in tabla.columns]
    if len(tabla.columns) != len(set(tabla.columns)):
        raise ValueError(f"Columnas duplicadas en {ruta.name}.")
    return tabla


def exigir_columnas(df: pd.DataFrame, columnas: list[str], nombre: str) -> None:
    faltantes = sorted(set(columnas) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas: {', '.join(faltantes)}.")


def convertir_numeros(df: pd.DataFrame, columnas: list[str], nombre: str) -> pd.DataFrame:
    df = df.copy()
    for col in columnas:
        if col not in df:
            continue
        original = df[col]
        valores = original
        if valores.dtype == object or isinstance(valores.dtype, pd.StringDtype):
            valores = valores.astype("string").str.strip().str.replace(",", ".", regex=False)
        convertidos = pd.to_numeric(valores, errors="coerce")
        invalidos = original.notna() & convertidos.isna() & original.astype(str).str.strip().ne("")
        if invalidos.any():
            ejemplos = original[invalidos].head(3).tolist()
            raise ValueError(f"{nombre}: valores no numéricos en {col}: {ejemplos}.")
        if np.isinf(convertidos.to_numpy(dtype=float, na_value=np.nan)).any():
            raise ValueError(f"{nombre}: {col} contiene valores infinitos.")
        df[col] = convertidos
    return df


def validar_ids(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    exigir_columnas(df, ["id_cand"], nombre)
    df = df.copy()
    if df["id_cand"].isna().any():
        raise ValueError(f"{nombre}: existen identificadores id_cand vacíos.")
    df["id_cand"] = df["id_cand"].astype(str).str.strip()
    if df["id_cand"].eq("").any():
        raise ValueError(f"{nombre}: existen identificadores id_cand vacíos.")
    dup = df.loc[df["id_cand"].duplicated(keep=False), "id_cand"].unique()
    if len(dup):
        raise ValueError(f"{nombre}: id_cand duplicados: {list(dup[:8])}. No se eliminaron registros.")
    return df


def preparar_candidatos(df: pd.DataFrame, audit: Auditoria) -> pd.DataFrame:
    df = validar_ids(df, "Tabla de candidatos")
    if df.empty:
        raise ValueError("La tabla de candidatos está vacía. Cargar una exportación con registros antes de generar figuras.")
    aliases = {"circularid": "circularidad", "blanqueami": "blanqueamiento",
               "distancia_": "distancia_similitud"}
    for corto, largo in aliases.items():
        if largo not in df and corto in df:
            df[largo] = df[corto]
    criticas = ["area_m2", "circularidad"] + ["dif_" + i for i in INDICES]
    exigir_columnas(df, criticas + ["prioridad_espectral"], "Tabla de candidatos")
    numericas = criticas + ["coord_x", "coord_y", "blanqueamiento", "vari", "radio_m",
        "distancia_similitud", "puntaje_indices", "puntaje_componentes"]
    numericas += ["cumple_" + i for i in INDICES]
    numericas += ["comp_" + c for c in ["vegetacion", "verdor", "humedad", "suelo"]]
    df = convertir_numeros(df, numericas, "Tabla de candidatos")
    faltantes = df[criticas].isna().sum()
    if faltantes.any():
        raise ValueError("Hay métricas esenciales sin datos: " + str(faltantes[faltantes > 0].to_dict()))
    if (df.area_m2 <= 0).any() or (df.circularidad < 0).any():
        raise ValueError("Las áreas deben ser positivas y la circularidad no debe ser negativa.")
    df["prioridad_espectral_original"] = df["prioridad_espectral"]
    df["prioridad_espectral"] = df["prioridad_espectral"].map(prioridad_normalizada)
    if "confianza" not in df:
        df["confianza"] = pd.NA
    if "confusor" not in df:
        df["confusor"] = pd.NA
    if "zona" not in df:
        df["zona"] = pd.NA
    df["confianza_visual"] = df["confianza"].map(confianza_normalizada)
    df["confusor_visual"] = df["confusor"].fillna("Sin información").astype(str).str.strip()
    df.loc[df.confusor_visual.eq(""), "confusor_visual"] = "Sin información"
    df["sector"] = df["zona"].map(lambda v: "Sin información" if pd.isna(v)
                                  else str(int(v)) if str(v).replace(".", "", 1).isdigit()
                                  and float(v).is_integer() else str(v).strip())
    df["diametro_equivalente_m_colab"] = 2 * np.sqrt(df.area_m2 / np.pi)
    audit.agregar("Identificadores de candidatos", "OK", f"{len(df)} registros; id_cand únicos y completos.")
    mayores = int(df.circularidad.gt(1).sum())
    audit.agregar("Rango superior de circularidad", "REVISAR" if mayores else "OK",
                  "Los valores superiores a 1 se conservan para revisión." if mayores
                  else "No hay valores superiores a 1.", mayores)
    if "radio_m" in df:
        ok = np.allclose(df.radio_m, np.sqrt(df.area_m2 / np.pi), atol=1e-5, rtol=1e-6)
        audit.comprobar("Radio circular equivalente", ok, "Comparación con sqrt(area_m2 / π).")
    for col in ["confianza_visual", "confusor_visual", "sector"]:
        n = int(df[col].eq("Sin información").sum())
        if n:
            audit.agregar(f"Datos de {col}", "REVISAR", "No se sustituyeron por una categoría observada.", n)
    extras = set(df.confianza_visual) - set(CONFIANZAS)
    if extras:
        audit.agregar("Categorías de confianza", "REVISAR", f"Se conservan categorías adicionales: {sorted(extras)}.")
    flags = ["cumple_" + i for i in INDICES]
    comps = ["comp_" + c for c in ["vegetacion", "verdor", "humedad", "suelo"]]
    presentes = [c for c in flags + comps if c in df]
    for col in presentes:
        if not df[col].isin([0, 1]).all():
            raise ValueError(f"{col}: los indicadores exportados deben ser 0 o 1 y estar completos.")
    if all(c in df for c in flags + comps):
        esperados = pd.DataFrame({
            "comp_vegetacion": df.cumple_NDVI * df.cumple_MSAVI,
            "comp_verdor": df.cumple_GLI, "comp_humedad": df.cumple_MSI,
            "comp_suelo": df.cumple_BSI}, index=df.index)
        for col in comps:
            n = int(df[col].ne(esperados[col]).sum())
            audit.comprobar(col, n == 0, "Verificación de la regla operativa de GEE_02.", n)
        p_indices = df[flags].sum(axis=1)
        p_componentes = esperados.sum(axis=1)
        for col, calculado in [("puntaje_indices", p_indices), ("puntaje_componentes", p_componentes)]:
            if col in df:
                n = int(df[col].ne(calculado).sum())
                audit.comprobar(col, n == 0, "Comprobación del puntaje exportado; no se reemplaza.", n)
        esperada = p_componentes.map({4: "Muy alta", 3: "Alta", 2: "Media", 1: "Baja", 0: "Sin señal fuerte"})
        n = int(df.prioridad_espectral.ne(esperada).sum())
        audit.comprobar("Prioridad espectral y componentes", n == 0,
                        "Se conserva la prioridad original y se comprueba su correspondencia con 0–4 componentes.", n)
    else:
        audit.agregar("Auditoría de puntajes", "REVISAR", "Faltan indicadores de cumplimiento o componentes; la prioridad se conserva sin reconstruirla.")
    return df


def preparar_contrastes(df: pd.DataFrame, indices: list[str], nombre: str) -> pd.DataFrame:
    columnas = ["dif_" + i for i in indices]
    exigir_columnas(df, ["grupo"] + columnas, nombre)
    df = convertir_numeros(df, columnas + ["radio_m"], nombre)
    df["grupo"] = df.grupo.astype(str).str.strip().str.upper()
    if not df.grupo.isin(["ANH", "QGIS"]).all():
        raise ValueError(f"{nombre}: grupo debe ser ANH o QGIS.")
    if df[columnas].isna().any().any():
        raise ValueError(f"{nombre}: hay contrastes sin datos; revisar los filtros de validez de GEE.")
    return df


def mismo_multiconjunto(a: pd.DataFrame, b: pd.DataFrame, columnas: list[str]) -> bool:
    """Compara conjuntos de mediciones sin asignar identidad a sus filas."""
    if len(a) != len(b):
        return False
    aa = np.round(a[columnas].to_numpy(float), 9)
    bb = np.round(b[columnas].to_numpy(float), 9)
    if len(a) == 0:
        return True
    orden_a = np.lexsort(tuple(aa[:, j] for j in reversed(range(aa.shape[1]))))
    orden_b = np.lexsort(tuple(bb[:, j] for j in reversed(range(bb.shape[1]))))
    return bool(np.allclose(aa[orden_a], bb[orden_b], rtol=0, atol=2e-8))


def posiciones(geometria: dict) -> list[tuple[float, float]]:
    tipo = geometria.get("type")
    coord = geometria.get("coordinates")
    if tipo == "GeometryCollection":
        return [p for g in geometria.get("geometries", []) for p in posiciones(g)]
    def caminar(v):
        if isinstance(v, (list, tuple)) and len(v) >= 2 and all(isinstance(x, (int, float)) for x in v[:2]):
            yield float(v[0]), float(v[1])
        elif isinstance(v, (list, tuple)):
            for item in v:
                yield from caminar(item)
    return list(caminar(coord))


def validar_geojson(obj: dict, nombre: str) -> None:
    if obj.get("type") != "FeatureCollection":
        raise ValueError(f"{nombre}: se requiere un GeoJSON FeatureCollection.")
    crs = obj.get("crs")
    if crs:
        descripcion = json.dumps(crs).upper()
        if not any(v in descripcion for v in ["4326", "CRS84", "WGS84", "WGS 84"]):
            raise ValueError(f"{nombre}: exportar el GeoJSON en WGS84 (EPSG:4326); no se infiere ni transforma el CRS.")
    for f in obj.get("features", []):
        pts = posiciones(f.get("geometry") or {})
        if not pts or any(not math.isfinite(x) or not math.isfinite(y) or
                          abs(x) > 180 or abs(y) > 90 for x, y in pts):
            raise ValueError(f"{nombre}: geometría vacía o coordenadas incompatibles con longitud/latitud WGS84.")


def centro_geometria(geometria: dict) -> tuple[float, float]:
    """Centro de muestreo de Point/Polygon/MultiPolygon; no recalcula área o C."""
    tipo = geometria.get("type")
    if tipo == "Point":
        return tuple(map(float, geometria["coordinates"][:2]))
    if tipo not in ["Polygon", "MultiPolygon"]:
        raise ValueError(f"Geometría de candidato no soportada: {tipo}. Usar puntos o áreas circulares equivalentes.")
    poligonos = [geometria["coordinates"]] if tipo == "Polygon" else geometria["coordinates"]
    ponderados = []
    for pol in poligonos:
        for j, anillo in enumerate(pol):
            a = np.asarray(anillo, dtype=float)[:, :2]
            if not np.array_equal(a[0], a[-1]):
                a = np.vstack([a, a[0]])
            origen = a[0].copy()
            a = a - origen  # reduce cancelación al operar con grados geográficos
            cruz = a[:-1, 0] * a[1:, 1] - a[1:, 0] * a[:-1, 1]
            area2 = cruz.sum()
            if abs(area2) < 1e-20:
                continue
            centro = np.array([((a[:-1, 0] + a[1:, 0]) * cruz).sum(),
                               ((a[:-1, 1] + a[1:, 1]) * cruz).sum()]) / (3 * area2) + origen
            peso = abs(area2) / 2 * (1 if j == 0 else -1)
            ponderados.append((centro, peso))
    if not ponderados or sum(p for _, p in ponderados) <= 0:
        raise ValueError("No se pudo calcular un centro de muestreo para la geometría.")
    return tuple(np.sum([c * p for c, p in ponderados], axis=0) / sum(p for _, p in ponderados))


def incorporar_geometria(df: pd.DataFrame, obj: dict, audit: Auditoria) -> pd.DataFrame:
    validar_geojson(obj, "GeoJSON de candidatos")
    filas = []
    for f in obj.get("features", []):
        prop = dict(f.get("properties") or {})
        prop["longitud_muestreo"], prop["latitud_muestreo"] = centro_geometria(f["geometry"])
        filas.append(prop)
    props = validar_ids(pd.DataFrame(filas), "GeoJSON de candidatos")
    if set(props.id_cand) != set(df.id_cand):
        faltan = sorted(set(df.id_cand) - set(props.id_cand))[:6]
        extras = sorted(set(props.id_cand) - set(df.id_cand))[:6]
        raise ValueError(f"CSV y GeoJSON tienen identificadores diferentes. Sin geometría: {faltan}; adicionales: {extras}.")
    p = props.set_index("id_cand").reindex(df.id_cand)
    numericas = ["area_m2", "circularidad"] + ["dif_" + i for i in INDICES]
    for col in numericas:
        alternativa = "circularid" if col == "circularidad" and col not in p else col
        if alternativa in p:
            valores = pd.to_numeric(p[alternativa], errors="coerce").to_numpy(float)
            ok = np.allclose(df[col].to_numpy(float), valores, rtol=1e-6, atol=1e-8, equal_nan=True)
            audit.comprobar(f"CSV–GeoJSON: {col}", ok, "Cruce uno a uno por id_cand.")
    for col in ["prioridad_espectral", "confianza", "confusor"]:
        if col in p and col in df:
            ok = np.array_equal(df[col].map(token).to_numpy(), p[col].map(token).to_numpy())
            audit.comprobar(f"CSV–GeoJSON: {col}", ok, "Comparación de atributos por identificador.")
    nuevas = ["longitud_muestreo", "latitud_muestreo"]
    nuevas += [c for c in props if (c.startswith("int_") or c.startswith("ext_")) and c not in df]
    props = convertir_numeros(props, nuevas, "GeoJSON de candidatos")
    integrado = df.merge(props[["id_cand"] + nuevas], on="id_cand", how="left", validate="one_to_one")
    for i in INDICES:
        a, b = "int_" + i, "ext_" + i
        if a in integrado and b in integrado:
            ok = np.allclose(integrado[a] - integrado[b], integrado["dif_" + i], atol=1e-8, equal_nan=True)
            audit.comprobar(f"Interior–entorno: {i}", ok, "El GeoJSON conserva medias interiores y exteriores compatibles con el contraste.")
    audit.agregar("Coordenadas para mapas", "OK", "Centros de las geometrías de muestreo en WGS84. coord_x/coord_y originales no se trataron como longitud/latitud.")
    return integrado


def cargar_datos(directorio: str | Path, archivos: dict | None = None,
                 contexto: dict | None = None) -> Datos:
    base = Path(directorio)
    nombres = ARCHIVOS_DEFAULT | (archivos or {})
    audit = Auditoria()
    fuentes = []
    tablas = {}
    objetos = {}
    for rol, nombre in nombres.items():
        if not nombre:
            continue
        ruta = Path(nombre) if Path(nombre).is_absolute() else base / nombre
        if not ruta.is_file():
            if rol == "candidatos":
                raise FileNotFoundError(f"Falta la tabla principal: {ruta.name}.")
            audit.agregar(f"Entrada opcional: {rol}", "INFO", f"No disponible: {ruta.name}; se omite su análisis específico.")
            continue
        fuentes.append({"rol": rol, "archivo": ruta.name, "bytes": ruta.stat().st_size,
                        "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest()})
        if ruta.suffix.lower() == ".csv":
            tablas[rol] = leer_csv(ruta)
        else:
            objetos[rol] = json.loads(ruta.read_text(encoding="utf-8-sig"))
    cand = preparar_candidatos(tablas["candidatos"], audit)
    contrastes = tablas.get("contrastes_5")
    if contrastes is not None:
        contrastes = preparar_contrastes(contrastes, INDICES, "Contrastes de cinco índices")
        qgis = contrastes.loc[contrastes.grupo.eq("QGIS")]
        ok = mismo_multiconjunto(cand, qgis, ["dif_" + i for i in INDICES])
        audit.comprobar("Tabla principal y contrastes QGIS", ok,
                        "Comparación conjunta de las cinco mediciones, sin asociar filas sin identificador.")
    amplios = tablas.get("contrastes_15")
    if amplios is not None:
        amplios = preparar_contrastes(amplios, INDICES_15, "Contrastes de quince índices")
        if contrastes is not None:
            for grupo in ["ANH", "QGIS"]:
                ok = mismo_multiconjunto(amplios.loc[amplios.grupo.eq(grupo)],
                                        contrastes.loc[contrastes.grupo.eq(grupo)], ["dif_" + i for i in INDICES])
                audit.comprobar(f"GEE_01–GEE_02: {grupo}", ok,
                                "Comparación de mediciones comunes para detectar versiones incompatibles.")
    resumen = tablas.get("distribuciones_5")
    if resumen is not None:
        exigir_columnas(resumen, ["grupo", "indice", "p25", "p75", "mean", "stdDev"], "Distribuciones GEE_02")
        resumen = convertir_numeros(resumen, ["p5", "p25", "p50", "p75", "p95", "mean", "stdDev"], "Distribuciones GEE_02")
        resumen["grupo"] = resumen.grupo.astype(str).str.upper().str.strip()
        resumen["indice"] = resumen.indice.astype(str).str.upper().str.strip()
        if resumen.duplicated(["grupo", "indice"]).any():
            raise ValueError("Distribuciones GEE_02: grupo/índice duplicados.")
        for i in INDICES:
            row = resumen.loc[resumen.grupo.eq("ANH") & resumen.indice.eq(i)]
            if len(row) != 1 or row[["p25", "p75"]].isna().any().any():
                raise ValueError(f"Falta un umbral ANH válido y único para {i}.")
            p = float(row.iloc[0]["p25" if i in ["NDVI", "MSAVI", "GLI"] else "p75"])
            if "cumple_" + i in cand:
                prueba = cand["dif_" + i].le(p) if i in ["NDVI", "MSAVI", "GLI"] else cand["dif_" + i].ge(p)
                n = int(prueba.astype(int).ne(cand["cumple_" + i]).sum())
                audit.comprobar(f"Umbral GEE_02: {i}", n == 0, f"Umbral original exportado: {p:.10g}.", n)
        if contrastes is not None:
            for _, row in resumen.iterrows():
                col = "dif_" + row.indice
                vals = contrastes.loc[contrastes.grupo.eq(row.grupo), col]
                ok = len(vals) > 0 and np.isclose(vals.mean(), row["mean"], atol=1e-8)
                ok = ok and np.isclose(vals.std(ddof=0), row["stdDev"], atol=1e-8)
                audit.comprobar(f"Distribuciones GEE: {row.grupo}/{row.indice}", bool(ok),
                                "Se comparan media y desviación poblacional. Los cuantiles GEE no se sustituyen por cuantiles NumPy.")
    morfo = tablas.get("circularidad_gee03")
    if morfo is not None:
        exigir_columnas(morfo, ["grupo", "n", "media", "desviacion_estandar", "minimo", "maximo"], "Resumen GEE_03")
        morfo = convertir_numeros(morfo, [c for c in morfo if c != "grupo"], "Resumen GEE_03")
        if morfo.grupo.duplicated().any():
            raise ValueError("Resumen GEE_03: grupos duplicados.")
        grupos = {"QGIS_total": cand}
        grupos.update({"QGIS_" + p: cand.loc[cand.prioridad_espectral.eq(p)] for p in PRIORIDADES})
        for grupo, subset in grupos.items():
            fila = morfo.loc[morfo.grupo.eq(grupo)]
            if not len(subset) and fila.empty:
                continue
            ok = len(fila) == 1
            if ok:
                fila = fila.iloc[0]
                ok = int(fila.n) == len(subset)
                if len(subset):
                    ok = ok and np.isclose(fila.media, subset.circularidad.mean(), atol=1e-6)
                    ok = ok and np.isclose(fila.minimo, subset.circularidad.min(), atol=1e-6)
                    ok = ok and np.isclose(fila.maximo, subset.circularidad.max(), atol=1e-6)
                    if len(subset) > 1:
                        ok = ok and np.isclose(fila.desviacion_estandar, subset.circularidad.std(ddof=1), atol=1e-6)
            audit.comprobar(f"GEE_03 y tabla actual: {grupo}", bool(ok),
                            "Se comprueban n, media, extremos y desviación muestral; no se exige igualdad de cuantiles aproximados.")
    geojson = objetos.get("geometria")
    if geojson is not None:
        cand = incorporar_geometria(cand, geojson, audit)
    contexto_capas = {k: v for k, v in objetos.items() if k != "geometria"}
    for nombre, obj in contexto_capas.items():
        validar_geojson(obj, nombre)
    return Datos(cand, contrastes, amplios, resumen, morfo, geojson, contexto_capas,
                 audit, fuentes, CONTEXTO_DEFAULT | (contexto or {}))


def describir(valores: pd.Series) -> dict:
    v = pd.to_numeric(valores, errors="coerce").dropna()
    salida = {"n": len(v), "media": v.mean(), "sd_muestral": v.std(ddof=1),
              "sd_poblacional": v.std(ddof=0), "minimo": v.min(), "maximo": v.max()}
    q = v.quantile([.05, .25, .5, .75, .95], interpolation="linear")
    salida.update({p: q.loc[t] for p, t in zip(["p5", "p25", "p50", "p75", "p95"], [.05, .25, .5, .75, .95])})
    if len(v):
        salida.update({"n_negativos": int(v.lt(0).sum()), "n_cero": int(v.eq(0).sum()),
                       "n_positivos": int(v.gt(0).sum()), "porcentaje_negativos": 100 * v.lt(0).mean()})
    return salida


def resumen_contrastes(df: pd.DataFrame, indices: list[str]) -> pd.DataFrame:
    filas = []
    for grupo in ["ANH", "QGIS"]:
        sub = df.loc[df.grupo.eq(grupo)]
        if sub.empty:
            continue
        for indice in indices:
            filas.append({"grupo": grupo, "indice": indice, **describir(sub["dif_" + indice])})
    return pd.DataFrame(filas)


def matriz_spearman(df: pd.DataFrame, columnas: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rangos promedio para empates. El análisis exige mediciones completas."""
    x = df[columnas]
    if x.isna().any().any():
        raise ValueError("Correlación: hay datos ausentes; revisar el conjunto de mediciones.")
    rho = x.rank(method="average").corr(method="pearson", min_periods=3)
    n = pd.DataFrame(len(x), index=columnas, columns=columnas)
    return rho, n


def crear_tablas(datos: Datos) -> dict[str, pd.DataFrame]:
    c = datos.candidatos
    tablas = {"01_candidatos_integrados": c.copy(), "00_auditoria": datos.auditoria.tabla()}
    conteo = c.prioridad_espectral.value_counts().reindex(PRIORIDADES, fill_value=0)
    tablas["02_conteo_prioridad"] = pd.DataFrame({"prioridad_espectral": PRIORIDADES,
        "n": conteo.to_numpy(), "porcentaje_total": conteo.to_numpy() / len(c) * 100})
    for campo, nombre in [("confianza_visual", "confianza"), ("confusor_visual", "confusor"), ("sector", "sector")]:
        x = c[campo].value_counts(dropna=False).rename_axis(campo).reset_index(name="n")
        x["porcentaje_total"] = x.n / len(c) * 100
        tablas["03_conteo_" + nombre] = x
    confs = [x for x in CONFIANZAS if x in set(c.confianza_visual)]
    confs += sorted(set(c.confianza_visual) - set(CONFIANZAS))
    cruce = pd.crosstab(c.prioridad_espectral, c.confianza_visual).reindex(index=PRIORIDADES, columns=confs, fill_value=0)
    tablas["04_cruce_prioridad_confianza_n"] = cruce.reset_index()
    denominador = cruce.sum(axis=1).replace(0, np.nan)
    tablas["04_cruce_prioridad_confianza_pct_fila"] = cruce.div(denominador, axis=0).mul(100).reset_index()
    tablas["04_cruce_prioridad_confusor_n"] = pd.crosstab(c.prioridad_espectral, c.confusor_visual).reindex(PRIORIDADES, fill_value=0).reset_index()
    filas = []
    metricas = ["area_m2", "circularidad", "diametro_equivalente_m_colab", "distancia_similitud"] + ["dif_" + i for i in INDICES]
    for prioridad in PRIORIDADES:
        sub = c.loc[c.prioridad_espectral.eq(prioridad)]
        for col in metricas:
            if col in c:
                filas.append({"prioridad_espectral": prioridad, "variable": col, **describir(sub[col])})
    tablas["05_metricas_por_prioridad_colab"] = pd.DataFrame(filas)
    contrastes5 = datos.contrastes_5
    if contrastes5 is None:
        contrastes5 = c[["dif_" + i for i in INDICES]].assign(grupo="QGIS")
    tablas["06_estadisticos_contrastes_5_colab"] = resumen_contrastes(contrastes5, INDICES)
    for n_indices, df, indices in [(5, contrastes5, INDICES), (15, datos.contrastes_15, INDICES_15)]:
        if df is None:
            continue
        if n_indices == 15:
            tablas["06_estadisticos_contrastes_15_colab"] = resumen_contrastes(df, indices)
        for grupo in ["ANH", "QGIS"]:
            sub = df.loc[df.grupo.eq(grupo)]
            if sub.empty:
                continue
            rho, n = matriz_spearman(sub, ["dif_" + i for i in indices])
            rho.index.name = "variable"
            n.index.name = "variable"
            tablas[f"07_spearman_{n_indices}_{grupo}"] = rho.reset_index()
            tablas[f"07_spearman_n_{n_indices}_{grupo}"] = n.reset_index()
    if datos.distribuciones_5 is not None:
        tablas["08_distribuciones_5_originales_GEE"] = datos.distribuciones_5.copy()
        filas = []
        for i in INDICES:
            row = datos.distribuciones_5.loc[datos.distribuciones_5.grupo.eq("ANH") & datos.distribuciones_5.indice.eq(i)].iloc[0]
            p = "p25" if i in ["NDVI", "MSAVI", "GLI"] else "p75"
            filas.append({"indice": i, "percentil_ANH": p, "umbral_original_GEE": row[p],
                          "condicion": "≤" if p == "p25" else "≥"})
        tablas["08_umbrales_originales_GEE"] = pd.DataFrame(filas)
    if datos.circularidad_gee03 is not None:
        tablas["09_circularidad_original_GEE03"] = datos.circularidad_gee03.copy()
    comps = ["comp_vegetacion", "comp_verdor", "comp_humedad", "comp_suelo"]
    if all(v in c for v in comps):
        filas = []
        for prioridad in PRIORIDADES:
            sub = c.loc[c.prioridad_espectral.eq(prioridad)]
            for col in comps:
                n = int(sub[col].eq(1).sum())
                filas.append({"prioridad_espectral": prioridad, "componente": col,
                              "n_grupo": len(sub), "n_cumple": n,
                              "porcentaje_grupo": n / len(sub) * 100 if len(sub) else np.nan})
        tablas["10_cumplimiento_componentes"] = pd.DataFrame(filas)
    for campo in ["confianza_visual", "confusor_visual", "sector"]:
        filas = []
        for grupo, sub in c.groupby(campo, sort=True, dropna=False):
            for i in INDICES:
                filas.append({campo: grupo, "indice": i, **describir(sub["dif_" + i])})
        tablas["11_contrastes_por_" + campo] = pd.DataFrame(filas)
    areas = datos.contexto.get("superficies_sector_km2") or {}
    if areas:
        filas = []
        for sector, n in c.sector.value_counts().items():
            if str(sector) not in {str(k) for k in areas}:
                raise ValueError(f"Falta la superficie confirmada del sector {sector}; no se infiere.")
            a = float(next(v for k, v in areas.items() if str(k) == str(sector)))
            if not math.isfinite(a) or a <= 0:
                raise ValueError("Las superficies de sector deben ser positivas y estar en km².")
            filas.append({"sector": sector, "n_candidatos": n, "superficie_km2_declarada": a, "candidatos_por_km2": n / a})
        tablas["12_densidad_operativa_por_sector"] = pd.DataFrame(filas)
    return tablas


def estilo_figuras() -> None:
    sns.set_theme(style="whitegrid", context="notebook", font="DejaVu Sans")
    plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
        "axes.titlesize": 12, "axes.labelsize": 10, "xtick.labelsize": 9,
        "ytick.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})


def formato_decimal(ax, eje="y"):
    fmt = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
    (ax.yaxis if eje == "y" else ax.xaxis).set_major_formatter(fmt)


def guardar_figura(fig, carpeta: Path, nombre: str, dpi: int = 300) -> str:
    carpeta.mkdir(parents=True, exist_ok=True)
    for extension in ["png", "pdf", "svg"]:
        fig.savefig(carpeta / f"{nombre}.{extension}", bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    return nombre


def cajas(ax, valores: list[pd.Series], etiquetas: list[str], colores: list[str]):
    activos = [(pd.to_numeric(v, errors="coerce").dropna().to_numpy(), e, c)
               for v, e, c in zip(valores, etiquetas, colores)]
    activos = [v for v in activos if len(v[0])]
    if not activos:
        ax.text(.5, .5, "Sin mediciones válidas", transform=ax.transAxes, ha="center")
        return
    bp = ax.boxplot([v[0] for v in activos], patch_artist=True, widths=.55,
        medianprops={"color": "#222222", "linewidth": 1.4},
        flierprops={"marker": ".", "markersize": 3, "alpha": .5})
    for p, (_, _, color) in zip(bp["boxes"], activos):
        p.set_facecolor(color)
        p.set_alpha(.8)
    ax.set_xticks(range(1, len(activos) + 1))
    ax.set_xticklabels([v[1] for v in activos])
    ax.grid(axis="x", visible=False)


def graficar_suite(datos: Datos, salida: Path, dpi: int = 300) -> list[str]:
    estilo_figuras()
    c = datos.candidatos
    carpeta = salida / "figuras"
    hechos = []
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
    for ax, campo, orden, colores, titulo in [
        (axes[0], "prioridad_espectral", PRIORIDADES, COLOR_PRIORIDAD, "Prioridad espectral"),
        (axes[1], "confianza_visual", [x for x in CONFIANZAS if x in set(c.confianza_visual)] +
         sorted(set(c.confianza_visual) - set(CONFIANZAS)), COLOR_CONFIANZA, "Confianza visual")]:
        counts = c[campo].value_counts().reindex(orden, fill_value=0)
        ax.barh(orden, counts, color=[colores.get(v, "#999999") for v in orden])
        ax.invert_yaxis()
        ax.set_xlim(0, max(1, counts.max()) * 1.28)
        for j, n in enumerate(counts):
            ax.text(n + max(1, counts.max()) * .02, j, f"{n} ({100*n/len(c):.1f} %)".replace(".", ","), va="center", fontsize=9)
        ax.set_xlabel("Número de candidatos")
        ax.set_title(titulo)
        ax.grid(axis="y", visible=False)
    fig.suptitle(f"Inventario de candidatos (n = {len(c)})", fontsize=14)
    hechos.append(guardar_figura(fig, carpeta, "01_prioridad_y_confianza", dpi))
    cruce = pd.crosstab(c.prioridad_espectral, c.confianza_visual).reindex(PRIORIDADES, fill_value=0)
    columnas = [v for v in CONFIANZAS if v in cruce] + sorted(set(cruce.columns) - set(CONFIANZAS))
    fig, ax = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
    sns.heatmap(cruce[columnas], annot=True, fmt="d", cmap="Blues", linewidths=.8,
                cbar_kws={"label": "Número de candidatos"}, ax=ax)
    ax.set(xlabel="Confianza visual", ylabel="Prioridad espectral",
           title="Cruce de clasificaciones visual y espectral")
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)
    hechos.append(guardar_figura(fig, carpeta, "02_cruce_prioridad_confianza", dpi))
    counts = c.confusor_visual.value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(8.5, max(3.5, len(counts)*.42)), layout="constrained")
    ax.barh(counts.index, counts, color="#547b8a")
    ax.set_xlim(0, max(1, counts.max()) * 1.12)
    for j, n in enumerate(counts):
        ax.text(n + max(1, counts.max())*.02, j, str(n), va="center")
    ax.set(xlabel="Número de candidatos", title="Categorías de confusor registradas en la revisión visual")
    ax.grid(axis="y", visible=False)
    hechos.append(guardar_figura(fig, carpeta, "03_confusores_visuales", dpi))
    counts = c.sector.value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6.5, 3.8), layout="constrained")
    ax.bar(["Sector " + v for v in counts.index], counts, color="#6b8c67")
    for j, n in enumerate(counts):
        ax.text(j, n + max(1, counts.max())*.025, str(n), ha="center")
    ax.set_ylim(0, max(1, counts.max()) * 1.17)
    ax.set(ylabel="Número de candidatos", title="Distribución del inventario por sector")
    hechos.append(guardar_figura(fig, carpeta, "04_conteo_por_sector", dpi))
    r = datos.contrastes_5
    if r is None:
        r = c[["dif_" + i for i in INDICES]].assign(grupo="QGIS")
    fig, axes = plt.subplots(2, 3, figsize=(11.6, 7.4), layout="constrained")
    for ax, i in zip(axes.flat, INDICES):
        grupos = [g for g in ["ANH", "QGIS"] if r.grupo.eq(g).any()]
        vals = [r.loc[r.grupo.eq(g), "dif_" + i] for g in grupos]
        etiquetas = [f"{g}\n(n = {len(v)})" for g, v in zip(grupos, vals)]
        cajas(ax, vals, etiquetas, [COLOR_GRUPO[g] for g in grupos])
        ax.axhline(0, color="#444444", ls="--", lw=.8)
        ax.set(title=i, ylabel="Contraste interior − entorno")
        formato_decimal(ax)
    axes.flat[-1].axis("off")
    axes.flat[-1].text(.05, .75, "Cajas: P25–P75\nLínea central: mediana\nBigotes: hasta 1,5 × RIQ\nPuntos: valores fuera de los bigotes\n\nCada índice conserva su escala.", va="top", fontsize=10)
    fig.suptitle("Distribución de los cinco contrastes espectrales", fontsize=14)
    hechos.append(guardar_figura(fig, carpeta, "05_contrastes_5_ANH_QGIS", dpi))
    for n_indices, df, indices in [(5, r, INDICES), (15, datos.contrastes_15, INDICES_15)]:
        if df is None:
            continue
        for grupo in ["ANH", "QGIS"]:
            sub = df.loc[df.grupo.eq(grupo)]
            if sub.empty:
                continue
            rho, _ = matriz_spearman(sub, ["dif_" + i for i in indices])
            rho.index = indices
            rho.columns = indices
            tam = (6.7, 5.7) if n_indices == 5 else (10.0, 8.8)
            fig, ax = plt.subplots(figsize=tam, layout="constrained")
            sns.heatmap(rho, vmin=-1, vmax=1, cmap="RdBu_r", square=True,
                annot=n_indices == 5, fmt=".2f", linewidths=.45,
                cbar_kws={"label": "ρ de Spearman"}, ax=ax)
            ax.set_title(f"Correlación de contrastes: {grupo} (n = {len(sub)})")
            ax.tick_params(axis="x", rotation=45)
            ax.tick_params(axis="y", rotation=0)
            hechos.append(guardar_figura(fig, carpeta, f"06_spearman_{n_indices}_{grupo}", dpi))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
    etiquetas = [v.replace("Sin señal fuerte", "Sin señal\nfuerte").replace("Muy alta", "Muy\nalta") for v in PRIORIDADES]
    for ax, campo, titulo, ylabel in [(axes[0], "circularidad", "Circularidad original", "Circularidad"),
                                    (axes[1], "area_m2", "Área de los objetos detectados", "Área (m²; eje logarítmico)")]:
        cajas(ax, [c.loc[c.prioridad_espectral.eq(v), campo] for v in PRIORIDADES], etiquetas,
               [COLOR_PRIORIDAD[v] for v in PRIORIDADES])
        ax.set(title=titulo, ylabel=ylabel, xlabel="Prioridad espectral")
        if campo == "area_m2":
            ax.set_yscale("log")
        else:
            formato_decimal(ax)
    hechos.append(guardar_figura(fig, carpeta, "07_morfometria_por_prioridad", dpi))
    fig, ax = plt.subplots(figsize=(8.6, 5.5), layout="constrained")
    for p in reversed(PRIORIDADES):
        sub = c.loc[c.prioridad_espectral.eq(p)]
        if len(sub):
            ax.scatter(sub.area_m2, sub.circularidad, s=26, color=COLOR_PRIORIDAD[p], alpha=.7, label=p)
    alta = c.loc[c.confianza_visual.eq("Alta")]
    if len(alta):
        ax.scatter(alta.area_m2, alta.circularidad, s=95, facecolors="none", edgecolors="black", lw=1.2,
                   label="Confianza visual alta")
    ax.set(xscale="log", xlabel="Área original (m²; eje logarítmico)", ylabel="Circularidad original",
           title="Área, circularidad y clasificaciones de los candidatos")
    formato_decimal(ax)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    hechos.append(guardar_figura(fig, carpeta, "08_area_circularidad", dpi))
    if datos.circularidad_gee03 is not None:
        ref = datos.circularidad_gee03
        sub = ref.loc[ref.grupo.isin(["ANH_total", "QGIS_total"])].set_index("grupo").reindex(["ANH_total", "QGIS_total"]).dropna(subset=["media"])
        if len(sub):
            fig, ax = plt.subplots(figsize=(6.6, 3.9), layout="constrained")
            y = np.arange(len(sub))
            ax.hlines(y, sub.minimo, sub.maximo, color="#b6b6b6", lw=3, label="Mínimo–máximo")
            ax.errorbar(sub.media, y, xerr=sub.desviacion_estandar, fmt="o", color="#24566b", capsize=5,
                        lw=1.6, label="Media ± DE muestral")
            ax.set_yticks(y)
            ax.set_yticklabels([f"{g.split('_')[0]} (n = {int(f.n)})" for g, f in sub.iterrows()])
            ax.invert_yaxis()
            ax.set(xlabel="Circularidad", title="Resumen de circularidad exportado de GEE_03")
            formato_decimal(ax, "x")
            ax.legend(loc="lower right", fontsize=8)
            hechos.append(guardar_figura(fig, carpeta, "09_circularidad_ANH_QGIS", dpi))
    interior = all("int_" + i in c and "ext_" + i in c for i in INDICES)
    if interior:
        fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.3), layout="constrained")
        for ax, i in zip(axes.flat, INDICES):
            x, y = c["ext_" + i], c["int_" + i]
            for conf in c.confianza_visual.unique():
                mask = c.confianza_visual.eq(conf)
                ax.scatter(x[mask], y[mask], s=18, alpha=.65, color=COLOR_CONFIANZA.get(conf, "#999999"), label=conf)
            extremos = np.concatenate([x.dropna().to_numpy(), y.dropna().to_numpy()])
            lo, hi = extremos.min(), extremos.max()
            margen = max((hi-lo)*.05, .005)
            ax.plot([lo-margen, hi+margen], [lo-margen, hi+margen], ls="--", color="#444444", lw=.8)
            ax.set_xlim(lo-margen, hi+margen)
            ax.set_ylim(lo-margen, hi+margen)
            ax.set(title=i, xlabel="Media en el entorno", ylabel="Media en el interior")
            formato_decimal(ax)
            formato_decimal(ax, "x")
        axes.flat[-1].axis("off")
        handles, labels = axes.flat[0].get_legend_handles_labels()
        axes.flat[-1].legend(handles, labels, title="Confianza visual", loc="upper left")
        axes.flat[-1].text(.05, .4, "Diagonal: interior = entorno\nArriba: contraste positivo\nAbajo: contraste negativo", va="top", fontsize=10)
        fig.suptitle("Mediciones interiores y exteriores de los candidatos", fontsize=14)
        hechos.append(guardar_figura(fig, carpeta, "10_interior_entorno_QGIS", dpi))
    if "longitud_muestreo" in c and "latitud_muestreo" in c:
        fig, ax = plt.subplots(figsize=(8.4, 6.5), layout="constrained")
        for p in reversed(PRIORIDADES):
            sub = c.loc[c.prioridad_espectral.eq(p)]
            if len(sub):
                ax.scatter(sub.longitud_muestreo, sub.latitud_muestreo, s=19, color=COLOR_PRIORIDAD[p], alpha=.8, label=p)
        ax.set(xlabel="Longitud (°)", ylabel="Latitud (°)", title="Localización de los candidatos por prioridad espectral")
        ax.set_aspect(1 / math.cos(math.radians(c.latitud_muestreo.mean())))
        ax.annotate("N", xy=(.93, .94), xytext=(.93, .79), xycoords="axes fraction",
                    ha="center", arrowprops={"arrowstyle": "-|>", "color": "black"}, fontsize=11)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
        formato_decimal(ax)
        formato_decimal(ax, "x")
        hechos.append(guardar_figura(fig, carpeta, "11_distribucion_espacial", dpi))
    return hechos


def serializable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return None
    raise TypeError(f"Tipo no serializable: {type(value).__name__}")


def crear_mapa_html(datos: Datos, salida: Path) -> Path | None:
    c = datos.candidatos
    if "longitud_muestreo" not in c:
        return None
    campos = ["id_cand", "sector", "prioridad_espectral", "confianza_visual", "confusor_visual",
              "area_m2", "circularidad", "longitud_muestreo", "latitud_muestreo"] + ["dif_" + i for i in INDICES]
    registros = json.loads(c[campos].to_json(orient="records", force_ascii=False, double_precision=12))
    datos_json = json.dumps({"registros": registros, "colores": COLOR_PRIORIDAD,
                            "geometrias": datos.geojson, "contexto": datos.capas_contexto}, ensure_ascii=False).replace("</", "<\\/")
    plantilla = r'''<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Mapa de candidatos</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>body{margin:0;font:14px system-ui;color:#25323c;background:#f6f8fa}header{padding:14px 20px;background:white;border-bottom:1px solid #ddd}h1{font-size:20px;margin:0 0 5px}.filters{display:flex;gap:10px;flex-wrap:wrap;padding:12px 20px}label{display:grid;gap:3px;font-size:12px}select,input{padding:7px;border:1px solid #bdc6ce;border-radius:4px;background:white}#map{height:590px}#status{margin:8px 20px}.tablewrap{max-height:330px;overflow:auto;margin:12px 20px}table{border-collapse:collapse;width:100%;font-size:12px;background:white}th,td{padding:7px;text-align:left;border-bottom:1px solid #eee}th{position:sticky;top:0;background:#e8eef2}tr[data-id]{cursor:pointer}tr[data-id]:hover{background:#f1f6f8}.legend{background:white;padding:10px;line-height:1.7;box-shadow:0 1px 5px #aaa}.dot{display:inline-block;width:11px;height:11px;border-radius:50%;margin-right:7px}.note{font-size:12px;color:#596773;margin:8px 20px;line-height:1.5}button{padding:7px 12px;border:1px solid #bdc6ce;border-radius:4px;cursor:pointer;background:white}</style>
</head><body><header><h1>Candidatos a círculos de hadas</h1><div>Prioridad espectral y revisión visual</div></header>
<div class="filters"><label>Prioridad<select id="prioridad"></select></label><label>Confianza visual<select id="confianza"></select></label><label>Confusor<select id="confusor"></select></label><label>Sector<select id="sector"></select></label><label>Buscar ID<input id="buscar" placeholder="C001"></label><label>&nbsp;<button id="restablecer">Restablecer filtros</button></label></div>
<div id="status"></div><div id="map"></div><div class="note">Los filtros afectan esta vista. La prioridad espectral y la confianza visual son atributos diferentes. Las áreas circulares corresponden al muestreo espectral. El mapa base y Leaflet requieren conexión a internet.</div>
<div class="tablewrap"><table><thead><tr><th>ID</th><th>Sector</th><th>Prioridad</th><th>Confianza</th><th>Confusor</th><th>Área (m²)</th><th>Circularidad</th></tr></thead><tbody id="tabla"></tbody></table></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
const D=__DATOS__;
const esc=v=>String(v??'').replace(/[&<>"']/g,x=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[x]));
const num=(v,n=3)=>v==null?'Sin información':Number(v).toLocaleString('es-CO',{maximumFractionDigits:n});
const map=L.map('map',{preferCanvas:true}).setView([12,-72],10);
const osm=L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{maxZoom:19,attribution:'© OpenStreetMap contributors'}).addTo(map);
const sat = L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', {maxZoom: 19, attribution: 'Tiles © Esri; fuentes indicadas por World Imagery'}).addTo(map);
const puntos=L.layerGroup().addTo(map),circulos=L.layerGroup();
const overlays={'Centros de muestreo':puntos,'Áreas circulares de muestreo':circulos};
const styles={area_estudio:{color:'#4d4d4d',weight:2,fillOpacity:0},poligonos_locales:{color:'#c29025',weight:2,fillOpacity:0},anh_geometria:{color:'#167a9e',weight:2,fillOpacity:.08}};
const names={area_estudio:'Área de estudio',poligonos_locales:'Polígonos locales',anh_geometria:'Registros ANH'};
for(const [k,geo] of Object.entries(D.contexto)){const layer=L.geoJSON(geo,{style:styles[k]||{},pointToLayer:(f,p)=>L.circleMarker(p,{radius:4,color:'#167a9e'})});overlays[names[k]||k]=layer;if(k!=='anh_geometria')layer.addTo(map);}
L.control.layers({'Imagen satelital': sat}, overlays, {collapsed: false}).addTo(map);
L.control.scale({imperial: false}).addTo(map);
const legend=L.control({position:'bottomright'});legend.onAdd=()=>{const el=L.DomUtil.create('div','legend');el.innerHTML='<b>Prioridad espectral</b><br>'+Object.entries(D.colores).map(([k,v])=>'<span class="dot" style="background:'+v+'"></span>'+esc(k)).join('<br>');return el;};legend.addTo(map);
const fields={prioridad:'prioridad_espectral',confianza:'confianza_visual',confusor:'confusor_visual',sector:'sector'};
for(const [id,k] of Object.entries(fields)){const el=document.getElementById(id);el.innerHTML='<option value="">Todas</option>'+[...new Set(D.registros.map(r=>String(r[k])))].sort().map(v=>'<option value="'+esc(v)+'">'+esc(v)+'</option>').join('');el.addEventListener('change',render);}
document.getElementById('buscar').addEventListener('input',render);
const geometry=new Map((D.geometrias?.features||[]).map(f=>[String(f.properties.id_cand).trim(),f]));
let markerById=new Map();
function popup(r){return '<b>'+esc(r.id_cand)+'</b><br>Prioridad: '+esc(r.prioridad_espectral)+'<br>Confianza visual: '+esc(r.confianza_visual)+'<br>Confusor: '+esc(r.confusor_visual)+'<br>Sector: '+esc(r.sector)+'<br>Área: '+num(r.area_m2,1)+' m²<br>Circularidad original: '+num(r.circularidad,4)+'<hr>'+['NDVI','MSAVI','GLI','MSI','BSI'].map(i=>'Δ '+i+': '+num(r['dif_'+i],5)).join('<br>');}
function render(){puntos.clearLayers();circulos.clearLayers();markerById=new Map();const buscar=document.getElementById('buscar').value.trim().toUpperCase();const registros=D.registros.filter(r=>Object.entries(fields).every(([id,k])=>!document.getElementById(id).value||String(r[k])===document.getElementById(id).value)&&(!buscar||r.id_cand.toUpperCase().includes(buscar)));for(const r of registros){const color=D.colores[r.prioridad_espectral];const m=L.circleMarker([r.latitud_muestreo,r.longitud_muestreo],{radius:r.confianza_visual==='Alta'?6:4,color:r.confianza_visual==='Alta'?'#111':color,weight:r.confianza_visual==='Alta'?2:1,fillColor:color,fillOpacity:.85}).bindPopup(popup(r)).addTo(puntos);markerById.set(r.id_cand,m);const f=geometry.get(r.id_cand);if(f&&f.geometry.type!=='Point')L.geoJSON(f,{style:{color,weight:1,fillOpacity:.13}}).bindPopup(popup(r)).addTo(circulos);}
document.getElementById('status').textContent='Se muestran '+registros.length+' de '+D.registros.length+' candidatos.';document.getElementById('tabla').innerHTML=registros.map(r=>'<tr data-id="'+esc(r.id_cand)+'"><td>'+esc(r.id_cand)+'</td><td>'+esc(r.sector)+'</td><td>'+esc(r.prioridad_espectral)+'</td><td>'+esc(r.confianza_visual)+'</td><td>'+esc(r.confusor_visual)+'</td><td>'+num(r.area_m2,1)+'</td><td>'+num(r.circularidad,4)+'</td></tr>').join('');}
document.getElementById('tabla').addEventListener('click',e=>{const row=e.target.closest('tr[data-id]');if(row){const m=markerById.get(row.dataset.id);map.setView(m.getLatLng(),17);m.openPopup();}});
document.getElementById('restablecer').onclick=()=>{for(const id of Object.keys(fields))document.getElementById(id).value='';document.getElementById('buscar').value='';render();};
render();if(D.registros.length){map.fitBounds(D.registros.map(r=>[r.latitud_muestreo,r.longitud_muestreo]),{padding:[25,25]});}
</script></body></html>'''
    ruta = salida / "mapa_interactivo.html"
    ruta.write_text(plantilla.replace("__DATOS__", datos_json), encoding="utf-8")
    return ruta


def escribir_reporte(datos: Datos, salida: Path, tablas: dict, figuras: list[str], mapa: Path | None) -> Path:
    c = datos.candidatos
    resumen = tablas["02_conteo_prioridad"]
    estilos = "body{font:15px system-ui;max-width:1080px;margin:35px auto;padding:0 20px;color:#24323c}h1{font-size:27px}h2{font-size:20px;margin-top:30px}img{max-width:100%;height:auto}table{border-collapse:collapse;font-size:13px;width:100%}td,th{padding:8px;border-bottom:1px solid #ddd;text-align:left}li{margin:8px 0}.note{color:#5e6a73}a{color:#25667d}"
    piezas = [f"<!doctype html><html lang='es'><meta charset='utf-8'><title>Análisis descriptivo</title><style>{estilos}</style>",
        "<h1>Análisis descriptivo de candidatos a círculos de hadas</h1>",
        f"<p>Sentinel-2: {html.escape(str(datos.contexto['periodo_sentinel2']))}. Inventario: {len(c)} candidatos.</p>",
        "<h2>Distribución de prioridades</h2>", resumen.to_html(index=False, float_format=lambda v: f"{v:.2f}".replace(".", ",")),
        "<h2>Definiciones del análisis</h2><ul>"]
    piezas += ["<li>" + html.escape(v) + "</li>" for v in NOTAS_METODO]
    piezas.append("</ul><h2>Tablas exportadas</h2><ul>")
    piezas += [f"<li><a href='tablas/{html.escape(v)}.csv'>{html.escape(v)}</a></li>" for v in tablas]
    piezas.append("</ul>")
    if mapa:
        piezas.append("<p><a href='mapa_interactivo.html'>Abrir mapa interactivo</a></p>")
    piezas.append("<h2>Figuras</h2>")
    for nombre in figuras:
        piezas.append(f"<figure><img src='figuras/{nombre}.png' alt='{html.escape(nombre)}'><figcaption><a href='figuras/{nombre}.pdf'>PDF</a> · <a href='figuras/{nombre}.svg'>SVG</a></figcaption></figure>")
    piezas.append("<h2>Auditoría de entradas</h2>" + datos.auditoria.tabla().to_html(index=False))
    piezas.append("<p class='note'>Las correlaciones y nuevos resúmenes se obtienen en esta etapa de Colab. No modifican los resultados originales de GEE.</p></html>")
    ruta = salida / "reporte.html"
    ruta.write_text("\n".join(piezas), encoding="utf-8")
    return ruta


def exportar_analisis(datos: Datos, salida: str | Path, dpi: int = 300,
                      tablas: dict | None = None) -> dict:
    destino = Path(salida)
    destino.mkdir(parents=True, exist_ok=True)
    datos.auditoria.tabla().to_csv(destino / "auditoria_entradas.csv", index=False, encoding="utf-8-sig")
    datos.auditoria.detener_si_errores()
    if tablas is None:
        tablas = crear_tablas(datos)
    (destino / "tablas").mkdir(exist_ok=True)
    for nombre, tabla in tablas.items():
        tabla.to_csv(destino / "tablas" / f"{nombre}.csv", index=False, encoding="utf-8-sig", float_format="%.12g")
    figuras = graficar_suite(datos, destino, dpi=dpi)
    mapa = crear_mapa_html(datos, destino)
    reporte = escribir_reporte(datos, destino, tablas, figuras, mapa)
    versiones = {}
    for p in ["numpy", "pandas", "matplotlib", "seaborn"]:
        versiones[p] = importlib.metadata.version(p)
    manifiesto = {"version_codigo": VERSION, "fecha_UTC": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(), "librerias": versiones, "contexto": datos.contexto,
        "n_candidatos": len(datos.candidatos), "fuentes": datos.fuentes, "notas_metodo": NOTAS_METODO,
        "convencion_cuantiles_colab": "interpolación lineal; original GEE preservado en tablas separadas",
        "convencion_desviacion_estandar": "sd_poblacional: ddof=0; sd_muestral: ddof=1",
        "figuras": figuras, "tablas": list(tablas)}
    (destino / "manifiesto_reproducibilidad.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"datos": datos, "tablas": tablas, "figuras": figuras,
            "directorio": destino, "mapa": mapa, "reporte": reporte}


def comprimir_resultados(directorio: str | Path) -> Path:
    directorio = Path(directorio)
    destino = directorio.with_suffix(".zip")
    with zipfile.ZipFile(destino, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(directorio.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(directorio))
    return destino






def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--entrada", default="datos")
    parser.add_argument("--salida", default="resultados")
    parser.add_argument("--dpi", type=int, default=300)
    args = parser.parse_args()
    datos = cargar_datos(args.entrada)
    print(datos.auditoria.tabla().to_string(index=False))
    resultado = exportar_analisis(datos, args.salida, dpi=args.dpi)
    print(f"Generadas {len(resultado['figuras'])} figuras y {len(resultado['tablas'])} tablas.")
    print(f"Resultados: {resultado['directorio'].resolve()}")



def nombre_entrada(nombre: str) -> str | None:
    """Identifica el rol del archivo sin cambiar su nombre original."""
    base = Path(nombre.replace("\\", "/")).name
    k = base.lower()
    roles_antiguos = {archivo.lower(): rol for rol, archivo in ARCHIVOS_DEFAULT.items()}
    if k in roles_antiguos:
        return roles_antiguos[k]
    patrones = [
        (r"^(gee_?02_)?tabla_clasificacion_espectral.*\.csv$", "candidatos"),
        (r"^(gee_?02_)?contrastes_indices_seleccionados.*\.csv$", "contrastes_5"),
        (r"^(gee_?02_)?distribuciones_indices_seleccionados.*\.csv$", "distribuciones_5"),
        (r"^(gee_?01_)?contrastes_indices_(?!seleccionados).*\.csv$", "contrastes_15"),
        (r"^gee_?03_resumen_circularidad.*\.csv$", "circularidad_gee03"),
        (r"^(gee_?02_)?candidatos_clasificacion_espectral.*\.geojson$", "geometria"),
    ]
    return next((rol for patron, rol in patrones if re.match(patron, k)), None)


def anillo_del_nombre(nombre: str) -> int | None:
    encontrado = re.search(r"(?:^|_)(30|60)m(?=[_.(]|$)", Path(nombre).name.lower())
    return int(encontrado.group(1)) if encontrado else None


def resolver_archivos(nombres: list[str], anillo_m: int | None = None) -> dict[str, str]:
    """Asocia roles con nombres reales y evita mezclar escenarios o versiones."""
    if anillo_m is not None and anillo_m not in (30, 60):
        raise ValueError("ANILLO_M debe ser 30 o 60.")
    reconocidos = [(nombre, nombre_entrada(nombre)) for nombre in nombres]
    reconocidos = [(nombre, rol) for nombre, rol in reconocidos if rol is not None]
    escenarios = {anillo_del_nombre(nombre) for nombre, _ in reconocidos}
    escenarios.discard(None)
    if len(escenarios) > 1:
        raise ValueError("Se recibieron archivos de 30 m y 60 m juntos. Carga solo un ZIP por ejecución.")
    if anillo_m is not None and escenarios and escenarios != {anillo_m}:
        real = next(iter(escenarios))
        raise ValueError(f"Seleccionaste {anillo_m} m, pero los archivos corresponden a {real} m. Cambia ANILLO_M o carga el ZIP correcto.")
    rutas = {}
    for nombre, rol in reconocidos:
        if rol in rutas:
            raise ValueError(f"Hay dos versiones del mismo tipo: {rutas[rol]} y {nombre}. Conserva una sola por ejecución.")
        rutas[rol] = nombre
    requeridos = ["candidatos", "contrastes_5", "contrastes_15",
                  "distribuciones_5", "circularidad_gee03", "geometria"]
    faltan = [rol for rol in requeridos if rol not in rutas]
    if faltan:
        raise FileNotFoundError("Faltan exportaciones: " + ", ".join(faltan) + ". Carga los seis archivos del escenario elegido.")
    return rutas


def reconocer_archivos(directorio: str | Path, anillo_m: int | None = None) -> dict[str, str]:
    """Encuentra las exportaciones originales para ejecución local."""
    base = Path(directorio)
    if not base.is_dir():
        raise FileNotFoundError(f"No existe la carpeta de datos: {base}. Extrae allí el ZIP del escenario elegido.")
    nombres = [str(p.relative_to(base)) for p in sorted(base.rglob("*")) if p.is_file()]
    return resolver_archivos(nombres, anillo_m)


def preparar_carga(archivos_subidos: dict[str, bytes], destino: str | Path,
                   anillo_m: int | None = None) -> dict:
    """Copia las exportaciones con sus nombres originales y sus bytes intactos."""
    destino = Path(destino)
    entradas = []
    ignorados = []
    total = 0
    for nombre, contenido in archivos_subidos.items():
        if nombre.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(contenido)) as z:
                for info in z.infolist():
                    if info.is_dir():
                        continue
                    if nombre_entrada(info.filename) is None:
                        ignorados.append(info.filename)
                        continue
                    if info.file_size > 100_000_000:
                        raise ValueError(f"Entrada demasiado grande: {info.filename}.")
                    total += info.file_size
                    if total > 200_000_000:
                        raise ValueError("Las entradas superan 200 MB. Este cuaderno trabaja con tablas y vectores.")
                    original = Path(info.filename.replace("\\", "/")).name
                    entradas.append((original, z.read(info), nombre + ":" + info.filename))
        else:
            if nombre_entrada(nombre) is None:
                ignorados.append(nombre)
                continue
            total += len(contenido)
            if total > 200_000_000:
                raise ValueError("Las entradas superan 200 MB. Este cuaderno trabaja con tablas y vectores.")
            original = Path(nombre.replace("\\", "/")).name
            entradas.append((original, contenido, nombre))
    rutas = resolver_archivos([original for original, _, _ in entradas], anillo_m)
    # Se valida toda la selección antes de escribir las copias.
    for original, data, _ in entradas:
        ruta = destino / original
        if ruta.exists() and ruta.read_bytes() != data:
            raise FileExistsError(f"{original} ya contiene otra versión. Usa una carpeta de carga nueva.")
    destino.mkdir(parents=True, exist_ok=True)
    encontrados = {}
    for original, data, origen in entradas:
        ruta = destino / original
        if not ruta.exists():
            ruta.write_bytes(data)
        encontrados[original] = {
            "rol": nombre_entrada(original), "origen": origen,
            "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
        }
    registro = {"anillo_m": anillo_m, "archivos": encontrados,
                "rutas_por_rol": rutas, "ignorados": ignorados}
    (destino / "registro_carga.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
    return registro


def verificar_contexto(datos: Datos) -> None:
    """Comprueba los parámetros cuando la exportación GEE_01 los incluye."""
    tabla = datos.contrastes_15
    if tabla is None:
        return
    for columna, parametro in [("anillo_m", "anillo_entorno_m"), ("escala_m", "escala_analisis_m")]:
        if columna in tabla:
            valores = pd.to_numeric(tabla[columna], errors="coerce")
            esperado = datos.contexto[parametro]
            ok = bool(valores.notna().all() and valores.eq(esperado).all())
            datos.auditoria.comprobar(
                f"Parámetro exportado: {columna}", ok,
                f"Se compara el dato de GEE con el valor seleccionado: {esperado} m.")



In [ ]:
#@title Parámetros del análisis
ANILLO_M = 30 #@param [30, 60] {type:"raw"}
if ANILLO_M not in (30, 60):
    raise ValueError("Selecciona 30 o 60 m.")

CONTEXTO = CONTEXTO_DEFAULT.copy()
CONTEXTO['periodo_sentinel2'] = '2024'
CONTEXTO['anillo_entorno_m'] = ANILLO_M
CONTEXTO['escala_analisis_m'] = 20
CONTEXTO['superficies_sector_km2'] = {}
DPI = 300

print('Escenario seleccionado:', ANILLO_M, 'm')
print('Carga únicamente los seis archivos de este escenario.')


## 2. Cargar las exportaciones originales

Selecciona un escenario en la celda anterior y carga **un solo ZIP**, con sus
seis exportaciones. También puedes seleccionar los seis CSV/GeoJSON directamente.
Los archivos conservan sus nombres originales y su contenido; se reconocen
automáticamente los prefijos GEE_01, GEE_02 y GEE03 y los sufijos de 30 o 60 m.

| Exportación | Uso |
|---|---|
| GEE_01_Contrastes_indices_[escenario].csv | Contrastes de los quince índices. |
| GEE_02_Tabla_clasificacion_espectral_[escenario].csv | Inventario principal. |
| GEE_02_Contrastes_indices_seleccionados_[escenario].csv | Contrastes de los cinco índices. |
| GEE_02_Distribuciones_indices_seleccionados_[escenario].csv | Distribuciones y umbrales de GEE. |
| GEE_02_Candidatos_clasificacion_espectral_[escenario].geojson | Geometrías y mediciones interiores/exteriores. |
| GEE03_Resumen_Circularidad_QGIS_ANH_[escenario].csv | Resumen morfométrico; acepta sufijos de descarga como (2). |

No cargues ambos escenarios juntos. Para ejecutar el otro, cambia ANILLO_M y
vuelve a ejecutar las celdas desde **Parámetros del análisis**, cargando su ZIP.
Cada ejecución crea carpetas separadas e identifica el ancho del anillo.

Opcionalmente puedes añadir area_estudio.geojson, poligonos_locales.geojson y
anh.geojson en WGS84 (EPSG:4326). Las coordenadas proyectadas coord_x/coord_y
se conservan y no se interpretan como grados.

Para ejecución local, extrae cada ZIP en datos/anillo_30m o datos/anillo_60m.


In [ ]:
#@title Subir un ZIP o los seis archivos del escenario elegido
try:
    EN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import files
    marca = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    ENTRADA = Path('/content/tesis_h2') / f'entrada_anillo_{ANILLO_M}m_{marca}'
    subidos = files.upload()
    registro = preparar_carga(subidos, ENTRADA, anillo_m=ANILLO_M)
    ARCHIVOS = registro['rutas_por_rol']
    print('Archivos originales reconocidos:')
    for nombre, info in registro['archivos'].items():
        print('  ', info['rol'], '→', nombre)
    if registro['ignorados']:
        print('Archivos adicionales omitidos:', registro['ignorados'])
else:
    ENTRADA = Path('datos') / f'anillo_{ANILLO_M}m'
    ARCHIVOS = reconocer_archivos(ENTRADA, anillo_m=ANILLO_M)
    print('Ejecución local. Datos:', ENTRADA.resolve())
    for rol, nombre in ARCHIVOS.items():
        print('  ', rol, '→', nombre)


## 3. Comprobar y cruzar los archivos

Se verifican identificadores, métricas, componentes y umbrales. El cruce del
CSV y el GeoJSON usa `id_cand` con relación uno a uno. Los contrastes sin ID
se comparan como conjuntos y se usan para resúmenes de grupo.

Un estado **ERROR** detiene la consolidación y exige revisar las versiones.
**REVISAR** identifica un dato que se conserva y necesita atención; **INFO**
señala una entrada opcional ausente. El informe de auditoría queda exportado.

In [ ]:
#@title Cargar datos y revisar auditoría
DATOS = cargar_datos(ENTRADA, ARCHIVOS, CONTEXTO)
verificar_contexto(DATOS)
from IPython.display import display, HTML, Image
display(DATOS.auditoria.tabla())
DATOS.auditoria.detener_si_errores()
print('Escenario:', DATOS.contexto['anillo_entorno_m'], 'm')
print('Candidatos conservados:', len(DATOS.candidatos))
if DATOS.contrastes_5 is not None:
    print('Mediciones por grupo:', DATOS.contrastes_5.grupo.value_counts().to_dict())


## 4. Resumir los resultados

Las tablas nuevas identifican media, mediana (P50), percentiles y las dos
convenciones de desviación estándar. Las distribuciones exportadas por GEE
se mantienen en tablas independientes y se emplean para auditar los umbrales.

Las correlaciones se calculan **por separado para ANH y QGIS**. Son un análisis
descriptivo añadido en Colab. No demuestran que cinco índices sean óptimos ni
que una señal espectral sea causada por hidrógeno. El cruce visual–espectral
es una tabla de correspondencias, no una matriz de exactitud de campo.

In [ ]:
#@title Generar tablas de análisis
TABLAS = crear_tablas(DATOS)
display(TABLAS['02_conteo_prioridad'])
display(TABLAS['04_cruce_prioridad_confianza_n'])
display(TABLAS['06_estadisticos_contrastes_5_colab'][
    ['grupo', 'indice', 'n', 'media', 'sd_muestral', 'sd_poblacional', 'p25', 'p50', 'p75']])
print('Tablas generadas:', len(TABLAS))


## 5. Generar figuras y exportar los productos

Las figuras se guardan en **PNG a 300 ppp, PDF y SVG**. Se incluyen conteos,
cruces de clasificaciones, distribuciones de contrastes, correlaciones,
comparaciones morfométricas y localización. El diámetro equivalente se calcula
en Colab como una variable derivada y se identifica con el sufijo `_colab`.

Se exportan tablas CSV, un reporte HTML, el mapa y un manifiesto con parámetros,
versiones de bibliotecas y huellas SHA-256 de las fuentes.

In [ ]:
#@title Exportar análisis completo
marca = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
nombre_salida = f'resultados_anillo_{ANILLO_M}m_{marca}'
SALIDA = Path('/content/tesis_h2') / nombre_salida if EN_COLAB else Path(nombre_salida)
RESULTADO = exportar_analisis(DATOS, SALIDA, dpi=DPI, tablas=TABLAS)
registro_carga = ENTRADA / 'registro_carga.json'
if registro_carga.exists():
    (SALIDA / 'registro_carga.json').write_bytes(registro_carga.read_bytes())
print('Escenario:', ANILLO_M, 'm')
print('Figuras:', len(RESULTADO['figuras']))
print('Tablas:', len(RESULTADO['tablas']))
print('Carpeta:', SALIDA)


In [ ]:
#@title Ver algunas figuras
for nombre in ['01_prioridad_y_confianza', '02_cruce_prioridad_confianza', '05_contrastes_5_ANH_QGIS']:
    ruta = SALIDA / 'figuras' / (nombre + '.png')
    if ruta.exists():
        display(Image(filename=str(ruta), width=1000))
print('Las demás figuras están en la carpeta figuras y en el ZIP final.')


## 6. Explorar candidatos y mapa

El mapa permite filtrar por prioridad, confianza, confusor, sector e ID.
Conserva todos los registros en el inventario exportado. Los contornos
circulares mostrados son unidades de muestreo; la circularidad del atributo
proviene del objeto original segmentado en QGIS. El mapa base requiere
internet. Puedes abrir `mapa_interactivo.html` después de descargar los resultados.

In [ ]:
#@title Consultar un candidato
ID_CANDIDATO = 'C001' #@param {type:"string"}
consulta = DATOS.candidatos.loc[DATOS.candidatos.id_cand.str.upper().eq(ID_CANDIDATO.strip().upper())]
if consulta.empty:
    print('ID no encontrado. Consulta la tabla del inventario.')
else:
    columnas = ['id_cand', 'sector', 'confianza_visual', 'confusor_visual', 'prioridad_espectral',
                'area_m2', 'circularidad', 'distancia_similitud'] + ['dif_' + i for i in INDICES]
    display(consulta[[c for c in columnas if c in consulta]].T)


In [ ]:
#@title Mostrar mapa interactivo
if RESULTADO['mapa'] is not None:
    contenido = RESULTADO['mapa'].read_text(encoding='utf-8')
    display(HTML('<iframe srcdoc="' + html.escape(contenido, quote=True) +
                 '" style="width:100%;height:900px;border:1px solid #ddd"></iframe>'))
else:
    print('No se cargó un GeoJSON compatible; se exportaron las tablas y figuras disponibles.')


## 7. Descargar todo

El ZIP de salida identifica el escenario de 30 o 60 m. Descomprímelo y abre
reporte.html para consultar los productos. El manifiesto conserva los nombres
originales y SHA-256 de las entradas; registro_carga.json documenta su carga.

Las tablas de la carpeta tablas son **productos derivados de Colab**. Pueden
incorporar columnas y estadísticos adicionales; sus números se escriben con
doce cifras significativas. Para el repositorio conserva también el ZIP de
entrada con las exportaciones originales intactas.


In [ ]:
#@title Descargar resultados
ZIP_RESULTADOS = comprimir_resultados(SALIDA)
if EN_COLAB:
    files.download(str(ZIP_RESULTADOS))
else:
    print('Archivo:', ZIP_RESULTADOS.resolve())


## Criterios de lectura

- Un contraste es una diferencia entre el interior y el entorno; no mide H₂.
- Una prioridad alta puede coexistir con un confusor o confianza visual baja.
- La ANH aportó registros de referencia y la calibración utilizó parte de ese
  antecedente; esta comparación no es una validación independiente de campo.
- Las figuras muestran las mediciones disponibles. No se eliminan valores
  extremos ni se inventan geometrías o superficies de sectores.
- La representación circular puede incluir coberturas que no pertenecían al
  contorno original. La ortoimagen y el compuesto Sentinel-2 también corresponden
  a fechas diferentes; considerar ambas condiciones al interpretar los resultados.

Documentación técnica: [Colab](https://research.google.com/colaboratory/intl/es/faq.html),
[cruces de pandas](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
y [Leaflet 1.9.4](https://leafletjs.com/reference).